In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override=True)

True

In [2]:
from accounts import Account

In [3]:
account = Account.get("Mit")
account

Account(name='mit', balance=10000.0, strategy='', holdings={}, transactions=[], portfolio_value_time_series=[])

In [4]:
account.buy_shares("AMZN", 50, "Because this bookstore website looks promising")

'Completed. Latest details:\n{"name": "mit", "balance": 9398.8, "strategy": "", "holdings": {"AMZN": 50}, "transactions": [{"symbol": "AMZN", "quantity": 50, "price": 12.024000000000001, "timestamp": "2025-10-29 11:53:36", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2025-10-29 11:53:36", 10448.8]], "total_portfolio_value": 10448.8, "total_profit_loss": 448.7999999999993}'

In [5]:
account.report()

'{"name": "mit", "balance": 9398.8, "strategy": "", "holdings": {"AMZN": 50}, "transactions": [{"symbol": "AMZN", "quantity": 50, "price": 12.024000000000001, "timestamp": "2025-10-29 11:53:36", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2025-10-29 11:53:36", 10448.8], ["2025-10-29 11:53:42", 10248.8]], "total_portfolio_value": 10248.8, "total_profit_loss": 248.79999999999927}'

In [6]:
account.list_transactions()

[{'symbol': 'AMZN',
  'quantity': 50,
  'price': 12.024000000000001,
  'timestamp': '2025-10-29 11:53:36',
  'rationale': 'Because this bookstore website looks promising'}]

### Now we write an MCP server and use it directly!

In [7]:
# Now let's use our accounts server as an MCP server

params = {"command": "uv", "args": ["run", "accounts_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()


In [8]:
mcp_tools

[Tool(name='get_balance', description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, annotations=None),
 Tool(name='get_holdings', description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, annotations=None),
 Tool(name='buy_shares', description="Buy shares of a stock.\n\n    Args:\n        name: The name of the account holder\n        symbol: The symbol of the stock\n        quantity: The quantity of shares to buy\n        rationale: The rationale for the purchase and fit with the account's strategy\n    ", inputSchema={'properties': {'name': {'title': 'Name', '

In [9]:
instructions = "You are able to manage an account for a client, and answer questions about the account."
request = "My name is Mit and my account is under the name Mit. What's my balance and my holdings?"
model = "gpt-4.1-mini"

In [10]:

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="account_manager", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("account_manager"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))


Mit, your current cash balance is $9,398.80. You have holdings of 50 shares in Amazon (AMZN). Is there anything specific you would like to do with your account?

### Now let's build our own MCP Client

In [11]:
from accounts_client import get_accounts_tools_openai, read_accounts_resource, list_accounts_tools

mcp_tools = await list_accounts_tools()
print(mcp_tools)
openai_tools = await get_accounts_tools_openai()
print(openai_tools)

[Tool(name='get_balance', description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, annotations=None), Tool(name='get_holdings', description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, annotations=None), Tool(name='buy_shares', description="Buy shares of a stock.\n\n    Args:\n        name: The name of the account holder\n        symbol: The symbol of the stock\n        quantity: The quantity of shares to buy\n        rationale: The rationale for the purchase and fit with the account's strategy\n    ", inputSchema={'properties': {'name': {'title': 'Name', 'ty

In [12]:
request = "My name is Ed and my account is under the name Ed. What's my balance?"

with trace("account_mcp_client"):
    agent = Agent(name="account_manager", instructions=instructions, model=model, tools=openai_tools)
    result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Ed, your current cash balance is $10,000. Is there anything specific you would like to do with your account?

In [15]:
context = await read_accounts_resource("Mit")
print(context)

{"name": "mit", "balance": 9398.8, "strategy": "", "holdings": {"AMZN": 50}, "transactions": [{"symbol": "AMZN", "quantity": 50, "price": 12.024000000000001, "timestamp": "2025-10-29 11:53:36", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2025-10-29 11:53:36", 10448.8], ["2025-10-29 11:53:42", 10248.8], ["2025-10-29 13:04:59", 14098.8], ["2025-10-29 15:51:43", 12898.8]], "total_portfolio_value": 12898.8, "total_profit_loss": 2898.7999999999993}


In [16]:
from accounts import Account
Account.get("Mit").report()

'{"name": "mit", "balance": 9398.8, "strategy": "", "holdings": {"AMZN": 50}, "transactions": [{"symbol": "AMZN", "quantity": 50, "price": 12.024000000000001, "timestamp": "2025-10-29 11:53:36", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2025-10-29 11:53:36", 10448.8], ["2025-10-29 11:53:42", 10248.8], ["2025-10-29 13:04:59", 14098.8], ["2025-10-29 15:51:43", 12898.8], ["2025-10-29 15:51:45", 13998.8]], "total_portfolio_value": 13998.8, "total_profit_loss": 3998.7999999999993}'